In [1]:
import os
import glob
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.signal as signal
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

In [2]:
sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.max_open_warning': 0})

### rPPG Extraction Algorithms

In [3]:
def extract_green_signal(rgb_frames, fps):
    """Method 1: Simple Green Channel Extraction"""
    green_sig = np.array([f[1] for f in rgb_frames]) # Green is index 1 in BGR
    return green_sig

In [4]:
def extract_chrom_signal(rgb_frames, fps):
    """Method 2: CHROM (Chrominance-based rPPG) - robust to motion/illumination changes"""
    rgb = np.array(rgb_frames, dtype=np.float64)
    # Normalize RGB traces by their mean
    r = rgb[:, 2] / (np.mean(rgb[:, 2]) + 1e-6)
    g = rgb[:, 1] / (np.mean(rgb[:, 1]) + 1e-6)
    b = rgb[:, 0] / (np.mean(rgb[:, 0]) + 1e-6)
    
    # Project into chrominance spaces
    x_chrom = 3.0 * r - 2.0 * g
    y_chrom = 1.5 * r + g - 1.5 * b
    
    # Bandpass filter components
    nyquist = 0.5 * fps
    b_filt, a_filt = signal.butter(2, [0.7 / nyquist, min(3.5 / nyquist, 0.99)], btype='band')
    x_f = signal.filtfilt(b_filt, a_filt, x_chrom)
    y_f = signal.filtfilt(b_filt, a_filt, y_chrom)
    
    alpha = np.std(x_f) / (np.std(y_f) + 1e-6)
    return x_f - alpha * y_f

In [5]:
def extract_pos_signal(rgb_frames, fps):
    """Method 3: POS (Plane Orthogonal to Skin)"""
    rgb = np.array(rgb_frames, dtype=np.float64)
    r = rgb[:, 2] / (np.mean(rgb[:, 2]) + 1e-6)
    g = rgb[:, 1] / (np.mean(rgb[:, 1]) + 1e-6)
    b = rgb[:, 0] / (np.mean(rgb[:, 0]) + 1e-6)
    
    # Projection vectors
    x_pos = r - b
    y_pos = g + b - 2.0 * r
    
    nyquist = 0.5 * fps
    b_filt, a_filt = signal.butter(2, [0.7 / nyquist, min(3.5 / nyquist, 0.99)], btype='band')
    x_f = signal.filtfilt(b_filt, a_filt, x_pos)
    y_f = signal.filtfilt(b_filt, a_filt, y_pos)
    
    alpha = np.std(x_f) / (np.std(y_f) + 1e-6)
    return x_f + alpha * y_f

### Feature Engineering

In [9]:
def get_signal_features(sig, fps):
    """Extracts statistical, spectral, and temporal features from any rPPG waveform"""
    if len(sig) < int(fps * 5):
        return [0.0] * 6
        
    nyquist = 0.5 * fps
    b_card, a_card = signal.butter(2, [0.7 / nyquist, min(3.5 / nyquist, 0.99)], btype='band')
    filtered = signal.filtfilt(b_card, a_card, sig)
    
    sig_mean = np.mean(sig)
    sig_variance = np.var(sig)
    filtered_std = np.std(filtered)
    
    # Spectral peak via Welch
    freqs, psd = signal.welch(filtered, fs=fps, nperseg=min(len(filtered), 512))
    dom_freq = freqs[np.argmax(psd)] if len(freqs) > 0 else 0.0
    
    # Derivative properties (Vascular dynamics proxy)
    first_deriv = np.gradient(filtered)
    sdppg_std = np.std(np.gradient(first_deriv))
    
    peaks, _ = signal.find_peaks(filtered, distance=int(fps*0.4))
    mean_peak_dist = np.mean(np.diff(peaks)) / fps if len(peaks) > 1 else 0.0
    
    return [sig_mean, sig_variance, filtered_std, dom_freq, sdppg_std, mean_peak_dist]

### Comparison/ Evaluation Pipeline

In [7]:
def run_extraction_method_comparison():
    df = pd.read_csv("db.csv")
    group_col = "patient_id" if "patient_id" in df.columns else "subject_id"
    if group_col not in df.columns and "file_id" in df.columns:
        group_col = "file_id"
        
    file_suffixes = [
        "_USBVideo_before", "_USBVideo_after",
        "_IriunWebcam_before", "_IriunWebcam_after",
        "_FullHDwebcam_before", "_FullHDwebcam_after"
    ]
    
    methods = {
        "Green Channel": extract_green_signal,
        "CHROM": extract_chrom_signal,
        "POS": extract_pos_signal
    }
    
    dataset_records = {m: {"X": [], "y_resp": [], "y_glu": [], "groups": []} for m in methods}
    
    print("Processing videos across extraction methods...")
    for idx, row in df.iterrows():
        raw_subj = row[group_col]
        subj_id = str(int(float(raw_subj))) if str(raw_subj).replace('.', '', 1).isdigit() else str(raw_subj).strip()
        
        resp_gt = row.get("respiratory", np.nan)
        hba1c_gt = row.get("glycated_hemoglobin", np.nan)
        
        if pd.isna(resp_gt) or pd.isna(hba1c_gt):
            continue
        glucose_gt = (28.7 * hba1c_gt) - 46.7
        
        # Collect per-subject data across matching video files
        subj_method_feats = {m: [] for m in methods}
        
        for suffix in file_suffixes:
            vid_paths = glob.glob(f"video/{subj_id}{suffix}.avi")
            if not vid_paths:
                continue
                
            cap = cv2.VideoCapture(vid_paths[0])
            fps = cap.get(cv2.CAP_PROP_FPS)
            fps = 30.0 if (fps == 0 or np.isnan(fps)) else fps
            
            rgb_frames = []
            while cap.isOpened():
                ret, frame = cap.read()
                if not ret or frame is None:
                    break
                h, w, _ = frame.shape
                roi = frame[int(h*0.15):int(h*0.85), int(w*0.2):int(w*0.8)]
                rgb_frames.append(np.mean(roi, axis=(0, 1))) # Mean B, G, R of ROI
            cap.release()
            
            if len(rgb_frames) < int(fps * 5):
                continue
                
            age = row.get("age", 25.0)
            bmi = row.get("bmi", 22.0)
            sex_val = 1.0 if str(row.get("sex", "M")).upper() == "M" else 0.0
            demographics = [age, bmi, sex_val]
            
            for m_name, m_func in methods.items():
                raw_sig = m_func(rgb_frames, fps)
                feats = get_signal_features(raw_sig, fps) + demographics
                subj_method_feats[m_name].append(feats)
                
        # Average across file views if available and append
        for m_name in methods:
            if subj_method_feats[m_name]:
                mean_feats = np.mean(subj_method_feats[m_name], axis=0)
                dataset_records[m_name]["X"].append(mean_feats)
                dataset_records[m_name]["y_resp"].append(resp_gt)
                dataset_records[m_name]["y_glu"].append(glucose_gt)
                dataset_records[m_name]["groups"].append(subj_id)

    # Evaluate via Leave-One-Group-Out (LOGO)
    logo = LeaveOneGroupOut()
    
    print("\n==================================================")
    print(" EXTRACTION METHOD BENCHMARK RESULTS (LOGO-CV)")
    print("==================================================")
    
    for m_name, data in dataset_records.items():
        if not data["X"]:
            print(f"\n--- Method: {m_name} (No valid samples found) ---")
            continue
            
        X = np.array(data["X"])
        y_resp = np.array(data["y_resp"])
        y_glu = np.array(data["y_glu"])
        groups = np.array(data["groups"])
        
        model_resp = RandomForestRegressor(n_estimators=100, random_state=42)
        resp_preds, resp_actuals = [], []
        for train_idx, test_idx in logo.split(X, y_resp, groups):
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X[train_idx])
            X_test = scaler.transform(X[test_idx])
            model_resp.fit(X_train, y_resp[train_idx])
            resp_preds.extend(model_resp.predict(X_test))
            resp_actuals.extend(y_resp[test_idx])
            
        model_glu = RandomForestRegressor(n_estimators=100, random_state=42)
        glu_preds, glu_actuals = [], []
        for train_idx, test_idx in logo.split(X, y_glu, groups):
            scaler = StandardScaler()
            X_train = scaler.fit_transform(X[train_idx])
            X_test = scaler.transform(X[test_idx])
            model_glu.fit(X_train, y_glu[train_idx])
            glu_preds.extend(model_glu.predict(X_test))
            glu_actuals.extend(y_glu[test_idx])
            
        print(f"\n--- Method: {m_name} ---")
        print(f"  [Respiratory Rate] MAE: {mean_absolute_error(resp_actuals, resp_preds):.4f} | RMSE: {np.sqrt(mean_squared_error(resp_actuals, resp_preds)):.4f}")
        print(f"  [Derived Glucose]  MAE: {mean_absolute_error(glu_actuals, glu_preds):.4f} | RMSE: {np.sqrt(mean_squared_error(glu_actuals, glu_preds)):.4f}")

In [8]:
if __name__ == "__main__":
    run_extraction_method_comparison()

Processing videos across extraction methods...

 EXTRACTION METHOD BENCHMARK RESULTS (LOGO-CV)

--- Method: Green Channel ---
  [Respiratory Rate] MAE: 1.4404 | RMSE: 1.9470
  [Derived Glucose]  MAE: 22.4555 | RMSE: 29.0521

--- Method: CHROM ---
  [Respiratory Rate] MAE: 1.5288 | RMSE: 1.9700
  [Derived Glucose]  MAE: 23.2097 | RMSE: 29.7921

--- Method: POS ---
  [Respiratory Rate] MAE: 1.5400 | RMSE: 1.9840
  [Derived Glucose]  MAE: 24.6642 | RMSE: 30.8734
